AI ASL Identifier
---
Paulina Almada Martínez - A01710029


## Preprocesado de datos

### Obtener imágenes

In [8]:
import kagglehub
import os
import shutil
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

Utilizamos la librería **Kagglehub** de Python para descargar las imágenes del dataset directamente de Kaggle, ya que son una cantidad considerable de archivos.

In [9]:
path = kagglehub.dataset_download('kapillondhe/american-sign-language')

100%|██████████| 4.64G/4.64G [00:41<00:00, 119MB/s]

Extracting files...


Después de obtener las imágenes, porque el dataset viene distribuido en test/train pero yo quiero incrementar la cantidad de imagenes de test, debemos primero pasar todas las imágenes a una nueva carpeta en memoria de Colab.

In [10]:
src = Path(path)
dst = Path("/content/asl_mixed") # nueva carpeta con las imágenes mezcladas
dst.mkdir(exist_ok=True)

El nombre de las imágenes es simplemente un número, con el conteo reseteando en cada carpeta. Por eso, como parte del proceso de extracción de las imágenes debemos obtener la categoría del nombre de la subcarpeta en la que está la imagen y cambiarle el nombre al label y número antes de guardarlas en nuevas carpetas de cada categoría con las originales de test y train combinadas. Esto, para facilitar trabajar con ellas más adelante y asegurar que no se sobreescriban al guardarlas.

In [11]:
for image_path in src.rglob("*"):
    if image_path.is_file() and image_path.suffix.lower() in {".jpg"}:
        label = image_path.parent.name # sacar categoría del nombre de carpeta
        label_dir = dst / label
        label_dir.mkdir(exist_ok=True) # crear nuevas carpetas por categoría con todas las imágenes
        label_name = f"{image_path.parent.parent.name}_{image_path.name}"
        shutil.copy2(image_path, label_dir / label_name)

print(f"Se encontraron {sum(len(files) for _, _, files in os.walk(dst))} imágenes")

Se encontraron 165782 imágenes


Después de crear las nuevas carpetas y renombrar las imágenes, metemos las imágenes correspondientes (basado en el label) en las nuevas carpetas.

In [12]:
for label in sorted(os.listdir(dst)):
    count = len(os.listdir(dst / label))
    print(f"{label}: {count} imágenes")

A: 6000 imágenes
B: 6000 imágenes
C: 6000 imágenes
D: 6000 imágenes
E: 6000 imágenes
F: 6000 imágenes
G: 6000 imágenes
H: 6000 imágenes
I: 6000 imágenes
J: 6000 imágenes
K: 6000 imágenes
L: 6000 imágenes
M: 6000 imágenes
N: 6000 imágenes
Nothing: 6000 imágenes
O: 6000 imágenes
P: 6000 imágenes
Q: 6000 imágenes
R: 5970 imágenes
S: 6000 imágenes
Space: 5890 imágenes
T: 5652 imágenes
U: 4546 imágenes
V: 6000 imágenes
W: 6000 imágenes
X: 6000 imágenes
Y: 5724 imágenes
Z: 6000 imágenes


### Split test/train/val

Ya que tenemos todas las imagenes juntas y etiquetadas, realizamos el split de test, train y validation. Ya que contamos con 165,782 imágenes, definí manejar un split de **80% train** (~ 132K en total, ~ 4.7K por categoría) y **10% validation y test** (~ 16.5K en total, ~ 589 por categoría). Considero que son suficientes imágenes para que el modelo aprenda y suficiente variedad al momento de probar para comprobar que sí está funcionando correctamente.

In [13]:
dst_split = Path("/content/asl_split") # nueva carpeta para guardarlas separadas

Después de separar las imágenes, creamos tres carpetas y guardamos las imágenes en cada una según corresponda.

In [14]:
for label in os.listdir(dst):
  images = os.listdir(dst / label)

  # primero separar test
  rest, test = train_test_split(images, test_size=0.10, random_state=42)

  # después separar val
  train, val = train_test_split(rest, test_size=0.10, random_state=42)

  for split_name, split_files in [("train", train), ("val", val), ("test", test)]:
    split_folder = dst_split / split_name / label
    split_folder.mkdir(parents=True, exist_ok=True)
    for f in split_files:
        shutil.copy2(dst / label / f, split_folder / f)

In [15]:
for split_name in ["train", "val", "test"]:
    for label in os.listdir(dst_split / split_name):
        count = len(os.listdir(dst_split / split_name / label))
        print(f"{split_name}/{label}: {count}")

train/X: 4860
train/I: 4860
train/V: 4860
train/L: 4860
train/B: 4860
train/M: 4860
train/C: 4860
train/H: 4860
train/D: 4860
train/P: 4860
train/J: 4860
train/G: 4860
train/U: 3681
train/S: 4860
train/A: 4860
train/Z: 4860
train/Y: 4635
train/N: 4860
train/T: 4577
train/E: 4860
train/Nothing: 4860
train/O: 4860
train/Q: 4860
train/K: 4860
train/R: 4835
train/Space: 4770
train/F: 4860
train/W: 4860
val/X: 540
val/I: 540
val/V: 540
val/L: 540
val/B: 540
val/M: 540
val/C: 540
val/H: 540
val/D: 540
val/P: 540
val/J: 540
val/G: 540
val/U: 410
val/S: 540
val/A: 540
val/Z: 540
val/Y: 516
val/N: 540
val/T: 509
val/E: 540
val/Nothing: 540
val/O: 540
val/Q: 540
val/K: 540
val/R: 538
val/Space: 531
val/F: 540
val/W: 540
test/X: 600
test/I: 600
test/V: 600
test/L: 600
test/B: 600
test/M: 600
test/C: 600
test/H: 600
test/D: 600
test/P: 600
test/J: 600
test/G: 600
test/U: 455
test/S: 600
test/A: 600
test/Z: 600
test/Y: 573
test/N: 600
test/T: 566
test/E: 600
test/Nothing: 600
test/O: 600
test/Q: 60

Guardamos las imágenes divididas en Google Drive para no tener que llevar a cabo todos los pasos anteriores cada vez que queremos correr el modelo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

shutil.copytree(dst_split, '/content/drive/MyDrive/TC3002B/asl_split')

### Normalización y carga de datos

Después de guardar las imágenes en Drive, podemos empezar futuras sesiones simplemente cargando las imágenes divididas.

In [7]:
from google.colab import drive
drive.mount('/content/drive')

dst_split = Path('/content/drive/MyDrive/TC3002B/asl_split')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Hasta este momento, por ser un dataset pesado, hemos estado trabajando solamente con los paths de las imágenes. Ya que están divididas, las cargamos. De paso, reducimos el tamaño de las imágenes para mejorar en tiempo de rendimiento del modelo y hacemos el escalamiento de los pixeles.

In [17]:
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen   = ImageDataGenerator(rescale=1./255)
test_datagen  = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    dst_split / "train",
    target_size=(64, 64),
    batch_size=32,
    class_mode="categorical"
)

val_generator = val_datagen.flow_from_directory(
    dst_split / "val",
    target_size=(64, 64),
    batch_size=32,
    class_mode="categorical"
)

test_generator = test_datagen.flow_from_directory(
    dst_split / "test",
    target_size=(64, 64),
    batch_size=32,
    class_mode="categorical"
)

Found 134278 images belonging to 28 classes.
Found 14924 images belonging to 28 classes.
Found 16580 images belonging to 28 classes.


## Primer modelo de clasificación

Para un primer acercamiento a nuestro clasificador de ASL, montamos una red densa simple. Ya que tenemos 28 categorías, montamos 28 neuronas de clasificación en nuestra última capa.

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense

In [19]:
def get_model_simple(input_shape):
    model = Sequential([
        Flatten(input_shape=input_shape),
        Dense(128, activation='relu'),
        Dense(28, activation='softmax')
    ])
    return model

In [20]:
def compile_model_simple(model):
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

In [21]:
def train_model_simple(model, train_generator, val_generator, epochs=5):
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=epochs
    )
    return history

Nuestro modelo termina teniendo la siguiente arquitectura:

In [22]:
model_nn = get_model_simple((64, 64, 3))
model_nn.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 12288)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,572,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 28)             │         3,612 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,576,604 (6.01 MB)

 Trainable params: 1,576,604 (6.01 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
compile_model_simple(model_nn)

In [24]:
history_nn = train_model_simple(model_nn, train_generator, val_generator)

Epoch 1/5
4197/4197 ━━━━━━━━━━━━━━━━━━━━ 313s 74ms/step - accuracy: 0.0383 - loss: 3.3406 - val_accuracy: 0.0362 - val_loss: 3.3312
Epoch 2/5
4197/4197 ━━━━━━━━━━━━━━━━━━━━ 249s 59ms/step - accuracy: 0.0356 - loss: 3.3313 - val_accuracy: 0.0362 - val_loss: 3.3312
Epoch 3/5
4197/4197 ━━━━━━━━━━━━━━━━━━━━ 219s 52ms/step - accuracy: 0.0356 - loss: 3.3313 - val_accuracy: 0.0362 - val_loss: 3.3311
Epoch 4/5
4197/4197 ━━━━━━━━━━━━━━━━━━━━ 213s 51ms/step - accuracy: 0.0364 - loss: 3.3313 - val_accuracy: 0.0362 - val_loss: 3.3313
Epoch 5/5
4197/4197 ━━━━━━━━━━━━━━━━━━━━ 211s 50ms/step - accuracy: 0.0358 - loss: 3.3313 - val_accuracy: 0.0362 - val_loss: 3.3311
